# Which fit counts the LoFTR matches best?

The fine stage (LoFTR with the MINIMA weights) gives pairs of matching points between the photo and a map window. A robust fit finds the shift / rotation / scale that most pairs agree with, and the number of agreeing pairs (the **inliers**) decides which candidate wins and whether we answer. Today that fit is OpenCV RANSAC.

This notebook runs the network **once** per query, keeps the raw matches, and then re-counts the inliers with several fits (`src/fine_matching/robust_fit.py`) on exactly the same matches. Everything after the network is offline, so a new fit or threshold costs seconds.

| method | what it is |
|---|---|
| RANSAC (current) | OpenCV similarity RANSAC, 3000 tries |
| LMEDS | OpenCV least median of squares: needs no pixel threshold to fit (the threshold only counts the inliers) |
| MAGSAC / MAGSAC++ (pymagsac) | picks pairs with MAGSAC on a homography, then a similarity is fitted to them |
| MAGSAC (OpenCV USAC) | the same idea with OpenCV's own MAGSAC |
| confidence-guided RANSAC | RANSAC that draws its sample pairs by the LoFTR confidence (the idea of NG-RANSAC / PROSAC, **without** a trained network) |

NG-RANSAC and DSAC (differentiable RANSAC) are not included: they are networks that must be trained on pairs with known poses, and there is no such training set here.

All methods end in a similarity and count inliers the same way (a pair is an inlier when it lies within `pixels` of the fitted similarity), so the counts are comparable. Because the counts differ between methods, **the answer bar must be compared per method**: the tables show the score at the current bar and at the best bar of each method.

# Part 1 · Settings

In [ ]:
import os, sys, math, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent                    # walk up to the repo folder
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

# ------------------------------------------------------------------ settings: change here, run again
N_PHOTOS = 500              # photos loaded for the data set
N_EVAL = 100                # photos evaluated
FIRST_PHOTO = 0             # first photo of the window
K_PEAKS = 50                # K per query (coarse shortlist for the DINO layer)
M_FINE = 10                 # M per query (places sent to LoFTR)
HEADINGS = [-5.0, 0.0, 5.0]     # yaw guesses; use [0.0] and [1.0] below for the plain single query
SCALE_FACTORS = [1.0, 1.05]
MATCHER_CONFIDENCE = 0.2    # LoFTR matches below this are dropped before the fit
PIXELS_LIST = [2.0, 3.0, 4.0]   # inlier distances tried for every method (3.0 is the current one)
PIXELS_MAIN = 3.0           # the distance used in the charts
MIN_INLIERS = 8             # the current answer bar
BARS = list(range(3, 31))   # bars tried offline
GAMMA = 1.0                 # choose by inliers + GAMMA * (DINO z-score)
SEED = 958476
plt.rcParams["figure.dpi"] = 90

# Part 2 · Set up and collect the LoFTR matches

The pipeline is the normal one (edge coarse search, DINO layer, LoFTR). For every photo and every query the raw matches of the M candidates are kept. This is the slow part; the fits below reuse it.

In [ ]:
from single_photo_pipeline import STAGES
from heading_query_pipeline import HeadingQueryPipeline
from src.header.scoring import points_for_error, final_score
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height
from train.coarse_methods.inputs import make_inputs
from semantic_rerank.decision import candidate_z
from semantic_rerank.dino_rerank import semantic_layer
from coarse_search.heading_queries import query_geometry
from fine_matching.loftr_matcher import candidate_matches, stage2_from_matches
from fine_matching.robust_fit import METHODS

pipeline = HeadingQueryPipeline("official", "edge_zncc", headings=HEADINGS, scale_factors=SCALE_FACTORS, k_peaks=K_PEAKS, m_fine=M_FINE, matcher="loftr_minima",
                                matcher_confidence=MATCHER_CONFIDENCE, gamma=GAMMA, min_inliers=MIN_INLIERS, satellite="new")
pipeline.setup(N_PHOTOS, SEED)
for line in pipeline.describe():
    print(line)
camera = pipeline.camera
test_set = pipeline.test_set
photo_ids = list(range(FIRST_PHOTO, min(FIRST_PHOTO + N_EVAL, len(test_set.items))))
print("%d photos, %d queries each" % (len(photo_ids), len(pipeline.queries)))

level 3: 0.85 m/pixel, picture 16378 x 16004, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11316, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%
map store ready, took 120 s
data: official | 379 photos available | zoom levels [3, 4, 5, 6, 7]
photos: official | map: new
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.20 | LoFTR coarse threshold 0.2 (default), upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
device: cuda
queries per photo: 6 (headings [-5.0, 0.0, 5.0], scale factors [1.0, 1.05]); K and M are per query
100 photos, 6 queries each


In [ ]:
import cv2
import torch
from header.settings import SEED as PIPELINE_SEED


def collect_query(item, prior, level, query):
    cv2.setRNGSeed(PIPELINE_SEED)
    geometry = query_geometry(query["heading"], query["scale_factor"])
    canvas, valid = render_query(item, camera, level, geometry)
    inputs = make_inputs(canvas, valid, level, prior, pipeline.families, pipeline.names)
    score_map = pipeline.coarse_method(inputs)
    peaks, details = semantic_layer(pipeline.encoder, score_map, inputs["geometry"], canvas, valid, level, pipeline.level_images,
                                    shortlist=pipeline.k_peaks, keep=pipeline.m_fine, weight=pipeline.dino_weight)
    z = candidate_z(peaks, details)
    stored = candidate_matches(pipeline.matcher, canvas, valid, level, peaks, pipeline.level_images)
    return dict(query, peaks=peaks, z=z, stored=stored)


collected = {}      # photo -> dict(level, queries=[dict(id, heading, scale_factor, peaks, z, stored)])
start = time.time()
for count, index in enumerate(photo_ids):
    item = test_set.items[index]
    prior = test_set.priors[index]
    level = level_for_height(item["rel_height"], camera)
    queries = []
    for query in pipeline.queries:
        # the GPU is shared with other notebooks: wait and try again when it is full
        for attempt in range(6):
            try:
                queries.append(collect_query(item, prior, level, query))
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print("GPU full at photo %d, waiting 30 s (try %d)" % (index, attempt + 1))
                time.sleep(30)
        else:
            raise RuntimeError("GPU stayed full: close the other notebooks that use it")
    collected[index] = dict(level=level, queries=queries)
    if (count + 1) % 10 == 0:
        print("photo %d/%d | %.0f s" % (count + 1, len(photo_ids), time.time() - start))
print("matches collected in %.0f s" % (time.time() - start))


photo 10/100 | 46 s
photo 20/100 | 86 s
photo 30/100 | 128 s
photo 40/100 | 175 s
photo 50/100 | 218 s
photo 60/100 | 265 s
photo 70/100 | 308 s
photo 80/100 | 352 s
photo 90/100 | 397 s
photo 100/100 | 441 s
matches collected in 441 s


# Part 3 · Count the inliers with every fit

For each method and each pixel distance: re-count the inliers of every candidate from the stored matches, pool the candidates of all queries, choose the answer (`inliers + GAMMA * z`, as in the pipeline) and keep the chosen candidate's inliers and its points if it were answered. The bar is applied afterwards, so one pass covers every bar.

In [ ]:
rows = []
fit_seconds = {}
for method_name, fit_function in METHODS.items():
    for pixels in PIXELS_LIST:
        clock = time.time()
        for index in photo_ids:
            item = test_set.items[index]
            level = collected[index]["level"]
            results = []
            for query in collected[index]["queries"]:
                stage2 = stage2_from_matches(query["stored"], level, MATCHER_CONFIDENCE, pixels, fit=fit_function)
                results.append(dict(id=query["id"], heading=query["heading"], peaks=query["peaks"], z=query["z"], stage2=stage2))
            answer = pipeline.answer_for(results, [query["id"] for query in results])
            truth_x, truth_y = test_set.truths[index]
            error_m = math.hypot(answer["x"] - truth_x, answer["y"] - truth_y)
            earned = points_for_error(error_m, item["rel_height"], camera.fx)
            rows.append(dict(method=method_name, pixels=pixels, photo=index, height=item["rel_height"], inliers=answer["inliers"], z_chosen=answer["z_chosen"],
                             error_m=error_m, error_sigma=error_m / test_set.sigmas[index], earned=earned))
        fit_seconds[(method_name, pixels)] = (time.time() - clock) / len(photo_ids)
        print("%-26s %.1f px: %.2f s per photo for the fits" % (method_name, pixels, fit_seconds[(method_name, pixels)]))
per_photo = pd.DataFrame(rows)

RANSAC (current)           2.0 px: 0.00 s per photo for the fits
RANSAC (current)           3.0 px: 0.00 s per photo for the fits
RANSAC (current)           4.0 px: 0.00 s per photo for the fits
LMEDS                      2.0 px: 0.00 s per photo for the fits
LMEDS                      3.0 px: 0.00 s per photo for the fits
LMEDS                      4.0 px: 0.00 s per photo for the fits
MAGSAC (pymagsac)          2.0 px: 0.02 s per photo for the fits
MAGSAC (pymagsac)          3.0 px: 0.01 s per photo for the fits
MAGSAC (pymagsac)          4.0 px: 0.01 s per photo for the fits
MAGSAC++ (pymagsac)        2.0 px: 0.01 s per photo for the fits
MAGSAC++ (pymagsac)        3.0 px: 0.01 s per photo for the fits
MAGSAC++ (pymagsac)        4.0 px: 0.01 s per photo for the fits
MAGSAC (OpenCV USAC)       2.0 px: 0.00 s per photo for the fits
MAGSAC (OpenCV USAC)       3.0 px: 0.00 s per photo for the fits
MAGSAC (OpenCV USAC)       4.0 px: 0.00 s per photo for the fits
confidence-guided RANSAC 

KeyboardInterrupt: 

# Part 4 · Results

For every method and distance: the contest score (out of 60) at the current bar, the best score over all bars and the bar that gives it, how many answered photos were wrong (−20 each, bar = current), the share of photos where the chosen place is within 4σ (`found`, bar-free), and the **AUC**: the chance that a photo with a right chosen place has more inliers than one with a wrong chosen place (how well the count separates right from wrong; 0.5 = useless, 1 = perfect).

In [ ]:
def score_at_bar(frame, bar):
    points = []
    for _, row in frame.iterrows():
        if row["inliers"] >= bar:
            points.append(row["earned"])
        else:
            points.append(0)
    return final_score(points)

def auc_of(frame):
    right = frame[frame["error_sigma"] <= 4]["inliers"].to_numpy()
    wrong = frame[frame["error_sigma"] > 4]["inliers"].to_numpy()
    if len(right) == 0 or len(wrong) == 0:
        return float("nan")
    wins = 0.0
    for value in right:
        wins += (value > wrong).sum() + 0.5 * (value == wrong).sum()
    return wins / (len(right) * len(wrong))

table = []
for (method_name, pixels), frame in per_photo.groupby(["method", "pixels"], sort=False):
    scores = {bar: score_at_bar(frame, bar) for bar in BARS}
    best_bar = max(scores, key=lambda bar: scores[bar])
    answered = frame[frame["inliers"] >= MIN_INLIERS]
    wrong_answers = int((answered["earned"] < 0).sum())
    table.append({"method": method_name, "pixels": pixels, "score at current bar": scores[MIN_INLIERS], "best score": scores[best_bar], "best bar": best_bar,
                  "answered": len(answered) / len(frame), "wrong answers (-20)": wrong_answers, "found <= 4 sigma": (frame["error_sigma"] <= 4).mean(),
                  "AUC": auc_of(frame), "median inliers (right)": frame[frame["error_sigma"] <= 4]["inliers"].median(),
                  "fit s per photo": fit_seconds[(method_name, pixels)]})
results_table = pd.DataFrame(table)
display(results_table.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for method_name in METHODS:
    frame = per_photo[(per_photo["method"] == method_name) & (per_photo["pixels"] == PIXELS_MAIN)]
    curve = [score_at_bar(frame, bar) for bar in BARS]
    axes[0].plot(BARS, curve, label=method_name)
axes[0].axvline(MIN_INLIERS, color="gray", ls="--")
axes[0].set_xlabel("answer bar (inliers)")
axes[0].set_ylabel("score /60")
axes[0].set_title("score against the bar, %.1f px" % PIXELS_MAIN)
axes[0].legend(fontsize=7)

main = results_table[results_table["pixels"] == PIXELS_MAIN]
axes[1].barh(main["method"], main["best score"], color="tab:blue", label="best bar")
axes[1].barh(main["method"], main["score at current bar"], color="tab:orange", height=0.4, label="current bar (%d)" % MIN_INLIERS)
axes[1].set_xlabel("score /60")
axes[1].set_title("best against current bar, %.1f px" % PIXELS_MAIN)
axes[1].legend()
plt.tight_layout()
plt.show()

## 4.1 Is the difference real?

100 photos give a score with a standard error of about 2.7 / 60, so a gap of one or two points between two fits is noise. The paired bootstrap below compares each method with RANSAC on the same photos, each with its own best bar from the table (so the comparison does not favour the bar RANSAC was tuned on).

In [ ]:
rng = np.random.default_rng(0)
base_name = "RANSAC (current)"
base_best_bar = int(results_table[(results_table["method"] == base_name) & (results_table["pixels"] == PIXELS_MAIN)]["best bar"].iloc[0])

def points_with_bar(frame, bar):
    result = np.zeros(len(frame))
    for position, (inliers, earned) in enumerate(zip(frame["inliers"], frame["earned"])):
        if inliers >= bar:
            result[position] = earned
    return result

base_frame = per_photo[(per_photo["method"] == base_name) & (per_photo["pixels"] == PIXELS_MAIN)].sort_values("photo")
base_points = points_with_bar(base_frame, base_best_bar)
compare = []
for method_name in METHODS:
    if method_name == base_name:
        continue
    frame = per_photo[(per_photo["method"] == method_name) & (per_photo["pixels"] == PIXELS_MAIN)].sort_values("photo")
    bar = int(results_table[(results_table["method"] == method_name) & (results_table["pixels"] == PIXELS_MAIN)]["best bar"].iloc[0])
    points = points_with_bar(frame, bar)
    gains = []
    for _ in range(2000):
        pick = rng.integers(0, len(points), len(points))
        gains.append(6 * (points[pick].sum() - base_points[pick].sum()) / (len(points)))
    compare.append({"method": method_name, "best bar": bar, "gain over RANSAC (/60)": 6 * (points.sum() - base_points.sum()) / len(points),
                    "95% low": np.percentile(gains, 2.5), "95% high": np.percentile(gains, 97.5)})
display(pd.DataFrame(compare).round(2))

# Part 5 · Save

In [ ]:
out_dir = ROOT / "tests" / "Fine_Fit_Methods"
out_dir.mkdir(parents=True, exist_ok=True)
tag = "official_edge_zncc_loftr_minima_q%d_K%d_M%d_n%d" % (len(pipeline.queries), K_PEAKS, M_FINE, len(photo_ids))
per_photo.to_csv(out_dir / ("per_photo_per_method_%s.csv" % tag), index=False)
results_table.to_csv(out_dir / ("summary_%s.csv" % tag), index=False)
print("saved to", out_dir)

**Caveats.** Same photos, same matches for every method, so the differences are due to the fit alone. The photos are the development set, and the bar for each method is chosen on the same photos it is then scored on (an optimistic number for all methods alike): a method that wins only by its best bar should be confirmed on other photos before it replaces RANSAC.